# Data preprocessing for the Twitter dataset

In [46]:
#BLOCK 1 (Imports & Setup)
# Import necessary libraries only
import os
import re
import string
import pandas as pd
import numpy as np
import emoji
import nltk
from langdetect import detect, LangDetectException
from tqdm.auto import tqdm
from nltk.corpus import stopwords
import spacy

# Download NLTK resources if not already present
nltk.download("punkt")
nltk.download("wordnet")
nltk.download('stopwords')

# Set random behavior for language detection
from langdetect import DetectorFactory
DetectorFactory.seed = 0

tqdm.pandas()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\malte\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\malte\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\malte\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [47]:
#BLOCK 2 (File Paths & Reading Raw Data)
# Define raw data directory
RAW_DIR = "raw"

# Read CSV files into DataFrames
musk_quote_tweets = pd.read_csv(os.path.join(RAW_DIR, "musk_quote_tweets.csv"))
all_musk_posts   = pd.read_csv(os.path.join(RAW_DIR, "all_musk_posts.csv"))


C:\Users\malte\AppData\Local\Temp\ipykernel_4636\3669752191.py:7: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  all_musk_posts   = pd.read_csv(os.path.join(RAW_DIR, "all_musk_posts.csv"))


In [48]:
#BLOCK 3 (Merge Quote Tweets)
# Create a combined text field for quotes and originals
musk_quote_tweets["quote_and_original"] = (
    musk_quote_tweets["musk_quote_tweet"].astype(str) +
    " // " +
    musk_quote_tweets["orig_tweet_text"].astype(str)
)

# Prepare for merge: only keep relevant columns
merge_df = musk_quote_tweets[["musk_tweet_id", "quote_and_original"]]

# Left merge quotes into full posts
all_musk_posts_with_quotes = (
    all_musk_posts
    .merge(merge_df, how="left", left_on="id", right_on="musk_tweet_id")
    .drop(columns=["musk_tweet_id"])
)

# Save intermediate merged CSV
output_path = os.path.join(RAW_DIR, "all_musk_posts_with_quotes.csv")
all_musk_posts_with_quotes.to_csv(output_path, index=False)
print(f"✅ Merged CSV saved to: {output_path}")

✅ Merged CSV saved to: raw\all_musk_posts_with_quotes.csv


In [49]:
#BLOCK 4 (Load & Filter by Date)
# Reload merged data with proper datetime parsing
musk_twitter_data = pd.read_csv(
    output_path,
    parse_dates=["createdAt"]
)

# Define time window with timezone-aware start date
START_DATE = pd.to_datetime("2015-01-01", utc=True)
END_DATE = musk_twitter_data["createdAt"].max()


# Keep only tweets within the specified date range
mask_time = (
    (musk_twitter_data["createdAt"] > START_DATE) &
    (musk_twitter_data["createdAt"] < END_DATE)
)
musk_twitter_data = musk_twitter_data.loc[mask_time]

# Normalize boolean-like fields and extract date
musk_twitter_data["isRetweet"] = musk_twitter_data["isRetweet"].astype(str).str.lower()
musk_twitter_data["possiblySensitive"] = musk_twitter_data["possiblySensitive"].astype(str).str.lower()
musk_twitter_data["fullText"] = musk_twitter_data["fullText"].astype(str)
#musk_twitter_data["date"] = musk_twitter_data["createdAt"].dt.date

print(f"Filtered data shape: {musk_twitter_data.shape}")

C:\Users\malte\AppData\Local\Temp\ipykernel_4636\799896567.py:3: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  musk_twitter_data = pd.read_csv(


Filtered data shape: (54023, 25)


In [50]:
    # BLOCK 4.1 – Zeitzonen-Umwandlung & Handelslogik-Shifting
#die zusätzlichen Spalten die hinzugefügt werden sind nur für die Nachvollziehbarkeit. spalte "date" bleibt die Spalte, die auch im weiteren verwendet wird, und spalte "date" wird auch entsprechend geshiftet. Zwar ändert der shift leider weniger als gedacht, da durch die vorige verwendung von UTC (+-0) (in gegensatz zu EDT (-4) was eigentlich sinnvoll gewesen wäre) der shift quasi schon vorher ungewollt umgesetzt wurde. aber wenigstens ists jetzt sauber und auch die winterzeit (-5) wird berücksichtigt. SOMIT ERGIBT SICH JETZT EIN VOLLSTÄNDIG SINNVOLLES SHIFTING: TWEETS, DIE NACH ENDE DER HANDELSZEIT (20:00) ABGESETZT WERDEN, WERDEN FÜR DIE PREDICTION DER FINANZMARKTTARGETS AM NÄCHSTEN TAG BERÜCKSICHTIGT.

# A) Original UTC-Zeit abspeichern
musk_twitter_data["datetime_original_UTC"] = musk_twitter_data["createdAt"]

# B) Umwandeln in US-Eastern Timezone (automatisch DST-aware)
musk_twitter_data["datetime_EDT"] = musk_twitter_data["createdAt"].dt.tz_convert("US/Eastern")

musk_twitter_data["datetime_EDT_shifted"] = (musk_twitter_data["datetime_EDT"] + pd.Timedelta(hours=4))

# C) Handelslogik-Anpassung (Shifting von 20:00 → Mitternacht durch +4h)
musk_twitter_data["date"] = (musk_twitter_data["datetime_EDT_shifted"]).dt.date

# sortieren nach datetime_original_UTC
musk_twitter_data = musk_twitter_data.sort_values(by="datetime_original_UTC").reset_index(drop=True)
#-----------------------------------------------------------------------------------------------------#

# Überprüfung: Wie viele Tweets wurden durch shift-logik auf den nächsten Tag "verschoben"?
original_date = musk_twitter_data["datetime_original_UTC"].dt.date
shifted_date = musk_twitter_data["date"]
num_shifted = (original_date != shifted_date).sum()
print(f"🕓 {num_shifted} Tweets wurden durch die Umzonung und den +4h-Shift einem anderen Kalendertag zugeordnet.")


🕓 855 Tweets wurden durch die Umzonung und den +4h-Shift einem anderen Kalendertag zugeordnet.


In [51]:
#BLOCK 5 (Emoji Demojization)
# Convert emojis to text representation in key columns
text_columns = ["fullText", "quote_and_original"]
for col in text_columns:
    musk_twitter_data[col] = (
        musk_twitter_data[col]
        .astype(str)
        .apply(lambda txt: emoji.demojize(txt, language="en"))
    )

In [52]:
#TODO @malte: missing values bei engagement index und den vier metrics irgendwie handeln

In [53]:
#BLOCK 6 (Engagement Index Calculation)
# Define engagement count columns and weights
count_cols = ['retweetCount', 'replyCount', 'likeCount', 'quoteCount']
weights = dict.fromkeys(count_cols, 0.25)

# Min-max normalize each count column
mins = musk_twitter_data[count_cols].min()
maxs = musk_twitter_data[count_cols].max()
normed = (musk_twitter_data[count_cols] - mins) / (maxs - mins)

# Compute weighted average for engagement index
weighted = normed * pd.Series(weights)
denominator = normed.notna() * pd.Series(weights)
musk_twitter_data['engagement_index'] = (
    weighted.sum(axis=1) / denominator.sum(axis=1)
).replace([np.inf, -np.inf], np.nan)


In [54]:
#BLOCK 7 (Prepare NLP DataFrames)
# Duplicate master DataFrame for separate processing
musk_twitter_data_all = musk_twitter_data.copy()

# Only NLP-specific DataFrame will be filtered and processed
musk_twitter_data_nlp = musk_twitter_data.copy()

In [55]:
#BLOCK 8 (Language Detection & Text Cleaning)
# Initialize NLP pipeline and stopwords
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
stop_words = set(stopwords.words('english'))

# Helper functions for cleaning

def safe_detect(text):
    """Detect language, return 'unknown' if error or too short."""
    text = str(text).strip()
    if len(text) < 10:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


def clean_basic(text):
    """Remove URLs, mentions, hashtags, and RT markers."""
    text = re.sub(r"http\S+|www\S+|@\w+|#|RT", "", str(text))
    return text.strip()


def clean_for_topic(text):
    """Lowercase, strip punctuation and digits for topic modeling."""
    text = clean_basic(text).lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    return re.sub(r"\d+", "", text)


def preprocess_lemmatized(text):
    """Lemmatize text and remove stopwords."""
    doc = nlp(clean_for_topic(text))
    return " ".join(
        token.lemma_ for token in doc
        if token.is_alpha and token.lemma_ not in stop_words
    )

# Boolean flag for quote tweets
musk_twitter_data_nlp['isQuote'] = (
    musk_twitter_data_nlp['isQuote']
    .astype(str)
    .str.lower()
    .eq('true')
)

# Apply language detection only on original tweets
mask_nonquote = (
    ~musk_twitter_data_nlp['isQuote'] &
    (musk_twitter_data_nlp['isRetweet'] != 'true')
)

musk_twitter_data_nlp.loc[mask_nonquote, 'language'] = (
    musk_twitter_data_nlp.loc[mask_nonquote, 'fullText']
    .progress_apply(safe_detect)
)

# Filter for tweets to keep: all quotes OR original English tweets with sufficient length
mask_keep = (
    musk_twitter_data_nlp['isQuote'] |
    (
        ~musk_twitter_data_nlp['isQuote'] &
        (musk_twitter_data_nlp['isRetweet'] != 'true') &
        (musk_twitter_data_nlp['fullText'].str.len() >= 10) &
        (musk_twitter_data_nlp['language'] == 'en')
    )
)

musk_twitter_data_nlp = musk_twitter_data_nlp.loc[mask_keep].reset_index(drop=True)

# Process raw and lemmatized text fields
for idx, row in tqdm(musk_twitter_data_nlp.iterrows(), total=len(musk_twitter_data_nlp)):
    if row['isQuote']:
        raw = row.get('quote_and_original', '')
        musk_twitter_data_nlp.at[idx, 'quote_and_original_raw'] = clean_basic(raw)
        musk_twitter_data_nlp.at[idx, 'quote_and_original_lemmatized'] = preprocess_lemmatized(raw)
    else:
        txt = row.get('fullText', '')
        musk_twitter_data_nlp.at[idx, 'text_raw'] = clean_basic(txt)
        musk_twitter_data_nlp.at[idx, 'text_lemmatized'] = preprocess_lemmatized(txt)

print(f"✅ NLP processing complete: {len(musk_twitter_data_nlp)} tweets retained.")

  0%|          | 0/45643 [00:00<?, ?it/s]

  0%|          | 0/45150 [00:00<?, ?it/s]

✅ NLP processing complete: 45150 tweets retained.


In [56]:
#BLOCK 9 (Save Cleaned Data)
# Define cleaned data directory
CLEAN_DIR = "cleaned"
os.makedirs(CLEAN_DIR, exist_ok=True)

# Export cleaned DataFrames to CSV
musk_twitter_data_nlp.to_csv(os.path.join(CLEAN_DIR, 'musk_twitter_data_nlp.csv'), index=False)
musk_twitter_data_all.to_csv(os.path.join(CLEAN_DIR, 'musk_twitter_data_all.csv'), index=False)

print("✅ Cleaned CSV files saved in 'cleaned/' directory.")

✅ Cleaned CSV files saved in 'cleaned/' directory.
